# Natural Language Processing — Foundations Notebook

This notebook walks through the core building blocks of modern NLP, from raw text
to language models and evaluation metrics. Each section pairs a short conceptual
explanation with runnable code on small, self-contained examples — no large
pretrained-model downloads required.

**Contents**
1. Text Processing Pipeline
2. Tokenization
3. Word Embeddings
4. Contextual Embeddings
5. Language Modeling
6. Evaluation Metrics

Libraries used: `nltk`, `scikit-learn`, `gensim`, `sentencepiece`, `numpy`.

In [1]:
# Install (uncomment if running outside this environment)
# !pip install nltk scikit-learn gensim sentencepiece numpy

import re
import math
import string
import random
from collections import Counter, defaultdict

import numpy as np

random.seed(42)
np.random.seed(42)

print("Environment ready.")

Environment ready.


In [2]:
# One-time NLTK data downloads (cached after first run)
import nltk

for pkg in [
    "punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker", "maxent_ne_chunker_tab", "words", "stopwords", "wordnet", "omw-1.4",
]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not fetch {pkg}: {e}")

print("NLTK resources ready.")

NLTK resources ready.


---
## 1. Text Processing Pipeline

Raw text is messy — mixed casing, punctuation, typos, HTML artifacts, and inconsistent
spacing. A **text processing pipeline** turns raw text into a clean, structured form that
downstream models can consume. A typical pipeline looks like:

```
Raw text → Cleaning → Normalization → Case folding → Tokenization/Segmentation →
Stop-word removal → Stemming/Lemmatization → POS tagging → NER → Parsing →
Feature extraction / Vectorization
```

Not every step is used in every pipeline — for example, deep learning models often skip
stop-word removal and stemming because subword tokenizers and contextual embeddings
already handle sparsity well. Classical ML pipelines (TF-IDF + Logistic Regression, etc.)
still benefit from aggressive normalization.

### 1.1 Cleaning & Normalization

**Cleaning** removes elements that carry no linguistic signal: HTML tags, URLs, emails,
extra whitespace, control characters. **Normalization** maps different surface forms to a
canonical form: unicode normalization (e.g. "café" vs "café" with combining accent),
expanding contractions ("don't" → "do not"), converting numbers to a standard form, etc.

**Regular expressions** are the primary tool for pattern-based cleaning.

In [3]:
raw_text = '''
<p>Hello!!! Visit https://example.com or email us at info@example.com 😀.</p>
I   can't   believe    it's   already   2026...    Call me at +91-9876543210 ASAP!!!
'''

def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)                      # strip HTML tags
    text = re.sub(r"http[s]?://\S+", " ", text)                 # strip URLs
    text = re.sub(r"\S+@\S+\.\S+", " ", text)                   # strip emails
    text = re.sub(r"[\U0001F300-\U0001FAFF]", " ", text)        # strip emoji
    text = re.sub(r"\+?\d[\d\-\s]{7,}\d", " ", text)             # strip phone-like numbers
    text = re.sub(r"\s+", " ", text).strip()                    # collapse whitespace
    return text

def expand_contractions(text: str) -> str:
    contractions = {
        "can't": "cannot", "won't": "will not", "n't": " not",
        "'re": " are", "'s": " is", "'ll": " will", "'ve": " have", "'m": " am",
    }
    for k, v in contractions.items():
        text = text.replace(k, v)
    return text

cleaned = clean_text(raw_text)
normalized = expand_contractions(cleaned)

print("CLEANED   :", cleaned)
print("NORMALIZED:", normalized)

CLEANED   : Hello!!! Visit or email us at . I can't believe it's already 2026... Call me at ASAP!!!
NORMALIZED: Hello!!! Visit or email us at . I cannot believe it is already 2026... Call me at ASAP!!!


### 1.2 Case Folding

**Case folding** (lower-casing, or the more unicode-robust `str.casefold()`) reduces
vocabulary size by treating "Apple", "apple", and "APPLE" as the same token. It is useful
for classical bag-of-words models but can *lose information* (e.g. "US" the country vs
"us" the pronoun, or named entities), which is why many modern tokenizers preserve case
and instead rely on subword units and large training corpora.

In [4]:
sample = "Apple released the new iPhone. Apple's stock, AAPL, rose 3%."
print("Original :", sample)
print("Lowered  :", sample.lower())
print("Casefold :", sample.casefold())

Original : Apple released the new iPhone. Apple's stock, AAPL, rose 3%.
Lowered  : apple released the new iphone. apple's stock, aapl, rose 3%.
Casefold : apple released the new iphone. apple's stock, aapl, rose 3%.


### 1.3 Stop-word Removal

**Stop words** are very high-frequency, low-information words ("the", "is", "and", ...).
Removing them shrinks the vocabulary and can improve classical bag-of-words models,
but is usually *not* done before feeding text into subword-tokenized neural models,
since stop words carry syntactic and sometimes semantic signal (e.g. negation "not").

In [5]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
tokens = "the quick brown fox is not jumping over the lazy dog".split()
filtered = [t for t in tokens if t.lower() not in stop_words]

print("All tokens     :", tokens)
print("Stop words     :", sorted(stop_words)[:10], "...")
print("After removal  :", filtered)

All tokens     : ['the', 'quick', 'brown', 'fox', 'is', 'not', 'jumping', 'over', 'the', 'lazy', 'dog']
Stop words     : ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an'] ...
After removal  : ['quick', 'brown', 'fox', 'jumping', 'lazy', 'dog']


### 1.4 Stemming vs. Lemmatization

Both reduce inflected words to a base form, but differently:

- **Stemming** chops off suffixes with heuristic rules (fast, crude, can produce
  non-words). Classic algorithm: **Porter Stemmer**.
- **Lemmatization** maps a word to its dictionary base form (*lemma*) using
  vocabulary + morphological analysis (slower, linguistically valid, needs POS info).

In [6]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "easily", "fairly", "studies", "better", "geese"]

def wordnet_pos(nltk_tag: str) -> str:
    if nltk_tag.startswith("J"):
        return wordnet.ADJ
    elif nltk_tag.startswith("V"):
        return wordnet.VERB
    elif nltk_tag.startswith("N"):
        return wordnet.NOUN
    elif nltk_tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged = pos_tag(words)

print(f"{'Word':<10}{'Stem':<10}{'Lemma':<10}")
for (w, tag) in tagged:
    stem = stemmer.stem(w)
    lemma = lemmatizer.lemmatize(w, pos=wordnet_pos(tag))
    print(f"{w:<10}{stem:<10}{lemma:<10}")

Word      Stem      Lemma     


running   run       run       
runs      run       run       
ran       ran       run       
easily    easili    easily    
fairly    fairli    fairly    
studies   studi     study     
better    better    well      
geese     gees      geese     


### 1.5 Sentence Segmentation

Splitting a document into sentences sounds trivial ("split on `.`") but periods are
ambiguous (abbreviations, decimals, ellipses). Real segmenters use statistical models
(e.g. NLTK's Punkt) trained to disambiguate sentence-final punctuation.

In [7]:
from nltk.tokenize import sent_tokenize

doc = ("Dr. Srivastav teaches AI at 9 a.m. He said, \"The model achieved 96.5% accuracy!\" "
       "Is that good enough? Let's find out...")

sentences = sent_tokenize(doc)
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")

1. Dr. Srivastav teaches AI at 9 a.m.
2. He said, "The model achieved 96.5% accuracy!"
3. Is that good enough?
4. Let's find out...


### 1.6 POS Tagging

**Part-of-Speech tagging** assigns a grammatical category (noun, verb, adjective, ...) to
each token. It's a sequence-labeling problem, classically solved with HMMs/CRFs, and now
usually with neural sequence taggers. NLTK ships a pretrained averaged-perceptron tagger
using the Penn Treebank tagset.

In [8]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize

sentence = "Dr. Srivastav is building a fall-risk prediction pipeline for eldercare robotics."
tokens = word_tokenize(sentence)
tags = pos_tag(tokens)

for tok, tag in tags:
    print(f"{tok:<15} {tag}")

Dr.             NNP
Srivastav       NNP
is              VBZ
building        VBG
a               DT
fall-risk       JJ
prediction      NN
pipeline        NN
for             IN
eldercare       NN
robotics        NNS
.               .


### 1.7 Named Entity Recognition (NER)

**NER** locates and classifies spans of text into predefined categories: `PERSON`,
`ORGANIZATION`, `GPE` (geo-political entity), `DATE`, `MONEY`, etc. Classically done with
CRFs over hand-crafted features; modern systems use fine-tuned transformer taggers
(e.g. BERT + a token-classification head).

In [9]:
from nltk import ne_chunk

sentence2 = "Gaurav Srivastav works at Krishna Institute of Engineering and Technology in Ghaziabad."
tokens2 = word_tokenize(sentence2)
tags2 = pos_tag(tokens2)
tree = ne_chunk(tags2)

print(tree)
print()
print("Detected entities:")
for subtree in tree:
    if hasattr(subtree, "label"):
        entity_text = " ".join(w for w, t in subtree.leaves())
        print(f"  {entity_text!r:40} -> {subtree.label()}")

(S
  (PERSON Gaurav/NNP Srivastav/NNP)
  works/VBZ
  at/IN
  (ORGANIZATION Krishna/NNP Institute/NNP)
  of/IN
  (GPE Engineering/NNP)
  and/CC
  (ORGANIZATION Technology/NNP)
  in/IN
  (GPE Ghaziabad/NNP)
  ./.)

Detected entities:
  'Gaurav Srivastav'                       -> PERSON
  'Krishna Institute'                      -> ORGANIZATION
  'Engineering'                            -> GPE
  'Technology'                             -> ORGANIZATION
  'Ghaziabad'                              -> GPE


In [10]:
import spacy
from spacy import displacy

# Load pretrained NER model (download if not present)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Input text
text = """
Dr. Gaurav works at KIET University in Ghaziabad.
He attended a conference in New Delhi in August 2026.
Google announced a new AI model worth $10 million.
"""

# Process text
doc = nlp(text)

# Print detected entities
print("Detected Entities:\n")

for ent in doc.ents:
    print(f"{ent.text:25} → {ent.label_:10} → {spacy.explain(ent.label_)}")

# Visualize
displacy.render(
    doc,
    style="ent",
    jupyter=True
)

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Detected Entities:

Gaurav                    → PERSON     → People, including fictional
KIET University           → ORG        → Companies, agencies, institutions, etc.
Ghaziabad                 → GPE        → Countries, cities, states
New Delhi                 → GPE        → Countries, cities, states
August 2026               → DATE       → Absolute or relative dates or periods
Google                    → ORG        → Companies, agencies, institutions, etc.
AI                        → GPE        → Countries, cities, states
$10 million               → MONEY      → Monetary values, including unit


### 1.8 Syntactic Parsing

**Parsing** builds a structural representation of a sentence.
- **Constituency parsing** groups words into nested phrases (NP, VP, ...).
- **Dependency parsing** links each word to a syntactic head, forming a tree of
  grammatical relations (subject, object, modifier, ...).

Below is a lightweight illustration using a hand-written context-free grammar (CFG)
for constituency parsing — production-grade systems use statistical/neural parsers
trained on large treebanks (e.g. the Penn Treebank).

In [11]:
import nltk

grammar = nltk.CFG.fromstring('''
  S -> NP VP
  NP -> Det N | Det N PP | 'I'
  VP -> V NP | V NP PP
  PP -> P NP
  Det -> 'the' | 'a'
  N -> 'model' | 'dataset' | 'researcher' | 'paper'
  V -> 'trained' | 'validated'
  P -> 'on' | 'with'
''')

parser = nltk.ChartParser(grammar)
sent = "I trained the model on a dataset".split()

for tree in parser.parse(sent):
    tree.pretty_print()

       S                                     
  _____|______________                        
 |                    VP                     
 |      ______________|________               
 |     |         |             PP            
 |     |         |          ___|___           
 |     |         NP        |       NP        
 |     |      ___|____     |    ___|_____     
 NP    V    Det       N    P  Det        N   
 |     |     |        |    |   |         |    
 I  trained the     model  on  a      dataset

       S                                 
  _____|__________                        
 |                VP                     
 |      __________|____                   
 |     |               NP                
 |     |      _________|___               
 |     |     |    |        PP            
 |     |     |    |     ___|___           
 |     |     |    |    |       NP        
 |     |     |    |    |    ___|_____     
 NP    V    Det   N    P  Det        N   
 |     |     |    |  

### 1.9 Feature Extraction & Vectorization Strategies

Once text is cleaned and tokenized, it must become **numeric features**. Classical
strategies include:

| Method | Idea | Notes |
|---|---|---|
| **One-hot / Bag-of-Words (BoW)** | Count of each vocabulary word per document | Sparse, ignores order |
| **TF-IDF** | Down-weights words common across all documents | Still sparse, but better discriminative weighting |
| **N-gram features** | Counts of contiguous word/char sequences | Captures local order |
| **Hashing trick** | Hash tokens into a fixed-size vector | Memory-efficient, no vocabulary storage |
| **Dense embeddings** (Sec. 3–4) | Learned low-dimensional vectors | Captures semantic similarity |

We compute BoW and TF-IDF below.

In [12]:
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = [
    "the model achieved high accuracy on the validation set",
    "the dataset was collected using wearable sensors",
    "high sensitivity is critical for fall risk prediction",
]

count_vec = CountVectorizer()
bow = count_vec.fit_transform(corpus)

tfidf_vec = TfidfVectorizer()
tfidf = tfidf_vec.fit_transform(corpus)

import pandas as pd
bow_df = pd.DataFrame(bow.toarray(), columns=count_vec.get_feature_names_out())
tfidf_df = pd.DataFrame(tfidf.toarray(), columns=tfidf_vec.get_feature_names_out()).round(2)

print("Bag-of-Words:")
display(bow_df)
print("\nTF-IDF:")
display(tfidf_df)

Bag-of-Words:


,accuracy,achieved,collected,critical,dataset,fall,for,high,is,model,...,prediction,risk,sensitivity,sensors,set,the,using,validation,was,wearable
0,1,1,0,0,0,0,0,1,0,1,...,0,0,0,0,1,2,0,1,0,0
1,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,1,1,0,1,1
2,0,0,0,1,0,1,1,1,1,0,...,1,1,1,0,0,0,0,0,0,0



TF-IDF:


,accuracy,achieved,collected,critical,dataset,fall,for,high,is,model,...,prediction,risk,sensitivity,sensors,set,the,using,validation,was,wearable
0,0.34,0.34,0.00,0.00,0.00,0.00,0.00,0.26,0.00,0.34,...,0.00,0.00,0.00,0.00,0.34,0.51,0.00,0.34,0.00,0.00
1,0.00,0.00,0.39,0.00,0.39,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.39,0.00,0.30,0.39,0.00,0.39,0.39
2,0.00,0.00,0.00,0.36,0.00,0.36,0.36,0.28,0.36,0.00,...,0.36,0.36,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.00


**Summary of Section 1:** raw text → clean/normalize → fold case → segment sentences →
tokenize words → (optionally) remove stop words → stem/lemmatize → POS tag → NER →
parse → vectorize. The next section zooms into tokenization, the step that most
directly determines a modern model's vocabulary and input representation.

---
## 2. Tokenization

**Tokenization** splits text into units ("tokens") that a model consumes. The choice of
granularity is a fundamental design decision:

| Granularity | Example (`"unbelievable"`) | Pros | Cons |
|---|---|---|---|
| **Word-level** | `["unbelievable"]` | Intuitive, interpretable | Huge vocab, OOV problem |
| **Character-level** | `["u","n","b",...]` | Tiny vocab, no OOV | Long sequences, weak semantics per unit |
| **Subword-level** | `["un", "believ", "able"]` | Balances vocab size & OOV | Slightly less interpretable |

Modern LLMs (GPT, BERT, T5, LLaMA, ...) all use **subword tokenization** — it's the
sweet spot between vocabulary size and sequence length, and it naturally handles rare
and unseen words by falling back to smaller pieces.

### 2.1 Word-level and Character-level Tokenization

In [13]:
text = "Unbelievable performance improvements!"

word_tokens = word_tokenize(text)
char_tokens = list(text.replace(" ", "_"))

print("Word-level tokens :", word_tokens)
print("Char-level tokens :", char_tokens)
print(f"\n#word tokens = {len(word_tokens)}, #char tokens = {len(char_tokens)}")

Word-level tokens : ['Unbelievable', 'performance', 'improvements', '!']
Char-level tokens : ['U', 'n', 'b', 'e', 'l', 'i', 'e', 'v', 'a', 'b', 'l', 'e', '_', 'p', 'e', 'r', 'f', 'o', 'r', 'm', 'a', 'n', 'c', 'e', '_', 'i', 'm', 'p', 'r', 'o', 'v', 'e', 'm', 'e', 'n', 't', 's', '!']

#word tokens = 4, #char tokens = 38


### 2.2 Byte Pair Encoding (BPE) — from scratch

BPE (Sennrich et al., 2016) starts with a vocabulary of individual characters and
**iteratively merges the most frequent adjacent symbol pair** into a new symbol, until a
target vocabulary size (or number of merges) is reached. This is exactly the algorithm
behind GPT-2/GPT-3's tokenizer.

Steps:
1. Represent each word as a sequence of characters + an end-of-word marker.
2. Count all adjacent symbol-pair frequencies across the corpus.
3. Merge the most frequent pair everywhere it occurs; add it to the vocabulary.
4. Repeat for a fixed number of merges.

In [14]:
def get_vocab(corpus):
    vocab = Counter()
    for word in corpus:
        symbols = tuple(list(word) + ["</w>"])
        vocab[symbols] += 1
    return vocab

def get_pair_freqs(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = pair
    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == bigram:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq
    return new_vocab

corpus = ["low", "lower", "lowest", "newest", "widest", "wide", "low", "newer"]
vocab = get_vocab(corpus)
merges = []
num_merges = 10

for step in range(num_merges):
    pairs = get_pair_freqs(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    print(f"Merge {step+1:2d}: {best_pair} (freq={pairs[best_pair]})")

print("\nLearned merges (in order):")
print(merges)
print("\nFinal symbol sequences:")
for symbols, freq in vocab.items():
    print(f"  {symbols}  (freq={freq})")

Merge  1: ('l', 'o') (freq=4)
Merge  2: ('lo', 'w') (freq=4)
Merge  3: ('e', 's') (freq=3)
Merge  4: ('es', 't') (freq=3)
Merge  5: ('est', '</w>') (freq=3)
Merge  6: ('low', '</w>') (freq=2)
Merge  7: ('e', 'r') (freq=2)
Merge  8: ('er', '</w>') (freq=2)
Merge  9: ('n', 'e') (freq=2)
Merge 10: ('ne', 'w') (freq=2)

Learned merges (in order):
[('l', 'o'), ('lo', 'w'), ('e', 's'), ('es', 't'), ('est', '</w>'), ('low', '</w>'), ('e', 'r'), ('er', '</w>'), ('n', 'e'), ('ne', 'w')]

Final symbol sequences:
  ('low</w>',)  (freq=2)
  ('low', 'er</w>')  (freq=1)
  ('low', 'est</w>')  (freq=1)
  ('new', 'est</w>')  (freq=1)
  ('w', 'i', 'd', 'est</w>')  (freq=1)
  ('w', 'i', 'd', 'e', '</w>')  (freq=1)
  ('new', 'er</w>')  (freq=1)


### 2.3 WordPiece

**WordPiece** (used by BERT) is similar to BPE but chooses the merge that **maximizes the
likelihood of the training corpus** under a unigram language model, rather than the raw
pair frequency. Conceptually:

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a) \times \text{count}(b)}$$

Merging the pair with the highest score favors merges where the combined unit is much
more likely than the two units occurring independently — i.e. it prefers merges that
reduce corpus "surprise" more than raw frequency alone. WordPiece also marks
non-initial subwords with a `##` prefix (e.g. `"##ing"`).

In [15]:
def wordpiece_scores(vocab):
    pair_freq = get_pair_freqs(vocab)
    symbol_freq = Counter()
    for symbols, freq in vocab.items():
        for s in symbols:
            symbol_freq[s] += freq
    scores = {}
    for (a, b), f_ab in pair_freq.items():
        scores[(a, b)] = f_ab / (symbol_freq[a] * symbol_freq[b])
    return scores

vocab_wp = get_vocab(corpus)
wp_merges = []
for step in range(8):
    scores = wordpiece_scores(vocab_wp)
    if not scores:
        break
    best_pair = max(scores, key=scores.get)
    vocab_wp = merge_vocab(best_pair, vocab_wp)
    wp_merges.append(best_pair)
    print(f"Merge {step+1}: {best_pair}  score={scores[best_pair]:.4f}")

def to_wordpiece_style(symbols):
    out = [symbols[0]]
    for s in symbols[1:]:
        out.append(("##" + s) if s != "</w>" else s)
    return out

print("\nExample WordPiece-style rendering:")
for symbols in list(vocab_wp.keys())[:4]:
    print(" ", to_wordpiece_style(list(symbols)))

Merge 1: ('i', 'd')  score=0.5000
Merge 2: ('s', 't')  score=0.3333
Merge 3: ('l', 'o')  score=0.2500
Merge 4: ('lo', 'w')  score=0.1250
Merge 5: ('w', 'id')  score=0.2500
Merge 6: ('e', 'r')  score=0.1250
Merge 7: ('w', 'er')  score=0.2500
Merge 8: ('low', 'er')  score=0.2500

Example WordPiece-style rendering:
  ['low', '</w>']
  ['lower', '</w>']
  ['low', '##e', '##st', '</w>']
  ['n', '##e', '##w', '##e', '##st', '</w>']


### 2.4 SentencePiece & the Unigram Language Model

**SentencePiece** is a tokenizer *framework* (used by T5, ALBERT, XLNet, LLaMA, ...) with
two key ideas:

1. It treats the input as a raw stream of unicode characters (space included, often shown
   as `▁`), so it is **language-agnostic** and needs no pre-tokenization/whitespace rules.
2. It supports the **Unigram Language Model** algorithm as an alternative to BPE:
   - Start from a large seed vocabulary of candidate subwords.
   - Assume each token in a sequence is generated independently: $P(x) = \prod_i p(x_i)$.
   - Use EM to estimate token probabilities and iteratively **prune** the vocabulary,
     removing tokens whose removal least hurts corpus likelihood, until the target
     vocabulary size is reached.
   - At inference, the segmentation with the highest probability (via Viterbi) is chosen
     — the model can therefore produce multiple valid segmentations for the same word.

Below we train an actual SentencePiece Unigram model on a toy corpus.

In [16]:
import sentencepiece as spm
import tempfile, os

toy_corpus = [
    "the model achieved high accuracy on the validation set",
    "the dataset was collected using wearable sensors",
    "high sensitivity is critical for fall risk prediction",
    "the transformer encodes contextual representations",
    "byte pair encoding builds a subword vocabulary",
    "sentencepiece treats whitespace as a normal symbol",
] * 20  # repeat so the tiny corpus has enough statistics

with tempfile.TemporaryDirectory() as d:
    corpus_path = os.path.join(d, "corpus.txt")
    with open(corpus_path, "w") as f:
        f.write("\n".join(toy_corpus))

    model_prefix = os.path.join(d, "sp_unigram")
    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix=model_prefix,
        vocab_size=60,
        model_type="unigram",
        character_coverage=1.0,
    )

    sp = spm.SentencePieceProcessor(model_file=model_prefix + ".model")

    test_sentence = "the transformer achieved high sensitivity"
    pieces = sp.encode(test_sentence, out_type=str)
    ids = sp.encode(test_sentence, out_type=int)

    print("Sentence :", test_sentence)
    print("Pieces   :", pieces)
    print("IDs      :", ids)
    print("Decoded  :", sp.decode(ids))

Sentence : the transformer achieved high sensitivity
Pieces   : ['▁the', '▁tr', 'a', 'n', 's', 'for', 'm', 'e', 'r', '▁ac', 'hi', 'e', 'v', 'ed', '▁high', '▁sens', 'iti', 'v', 'it', 'y']
IDs      : [14, 33, 18, 49, 4, 35, 16, 5, 7, 36, 27, 5, 19, 44, 25, 28, 38, 19, 50, 8]
Decoded  : the transformer achieved high sensitivity


### 2.5 Vocabulary Construction, Special Tokens, Token IDs, and Attention Masks

A trained tokenizer defines a **vocabulary**: a fixed mapping `token -> integer id`.
On top of the "content" vocabulary, models reserve **special tokens**, e.g.:

| Token | Purpose |
|---|---|
| `[PAD]` / `<pad>` | Pads shorter sequences to a common length |
| `[CLS]` / `<s>` | Sequence-level representation (classification) |
| `[SEP]` / `</s>` | Marks sentence/segment boundaries |
| `[MASK]` | Placeholder for masked-language-model pretraining |
| `[UNK]` / `<unk>` | Out-of-vocabulary fallback |

When batching sequences of different lengths, an **attention mask** (1 = real token,
0 = padding) tells the model which positions to ignore in self-attention.

In [17]:
vocab = {
    "[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4,
    "the": 5, "model": 6, "achieved": 7, "high": 8, "accuracy": 9,
}

def encode_batch(sentences, vocab, max_len=8):
    input_ids, attention_masks = [], []
    for sent in sentences:
        toks = ["[CLS]"] + sent.split() + ["[SEP]"]
        ids = [vocab.get(t, vocab["[UNK]"]) for t in toks]
        mask = [1] * len(ids)
        # pad or truncate to max_len
        if len(ids) < max_len:
            pad_len = max_len - len(ids)
            ids += [vocab["[PAD]"]] * pad_len
            mask += [0] * pad_len
        else:
            ids, mask = ids[:max_len], mask[:max_len]
        input_ids.append(ids)
        attention_masks.append(mask)
    return np.array(input_ids), np.array(attention_masks)

batch = ["the model achieved high accuracy", "the model was excellent"]
ids, masks = encode_batch(batch, vocab)

print("Vocab:", vocab)
print("\nInput IDs:\n", ids)
print("\nAttention masks:\n", masks)

Vocab: {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4, 'the': 5, 'model': 6, 'achieved': 7, 'high': 8, 'accuracy': 9}

Input IDs:
 [[2 5 6 7 8 9 3 0]
 [2 5 6 1 1 3 0 0]]

Attention masks:
 [[1 1 1 1 1 1 1 0]
 [1 1 1 1 1 1 0 0]]


### 2.6 Out-of-Vocabulary (OOV) Handling Strategies

Even with subword tokenization, OOV problems can arise (e.g. unseen unicode scripts,
emoji, typos). Common strategies:

1. **`[UNK]` fallback** — map anything unrecognized to a single UNK token (loses info).
2. **Byte-level fallback** (GPT-2 style) — operate on raw UTF-8 bytes, so *every* string
   is representable (256 base byte tokens); there is no possible OOV at the byte level.
3. **Character-level fallback** — subword tokenizer backs off to characters when no
   known subword matches.
4. **Subword decomposition** — split unseen words into smaller *known* pieces
   (`"unhappiness"` → `"un", "happi", "ness"`), the core BPE/WordPiece/Unigram idea.

The cell below shows a WordPiece-like greedy longest-match-first segmentation, the
strategy BERT's tokenizer uses at inference time to segment unseen words.

In [18]:
known_subwords = {"un", "happi", "##ness", "##ly", "run", "##ning", "the", "model"}

def greedy_wordpiece_segment(word, vocab, unk_token="[UNK]"):
    if word in vocab:
        return [word]
    pieces = []
    start = 0
    while start < len(word):
        end = len(word)
        matched = None
        while end > start:
            piece = word[start:end]
            candidate = piece if start == 0 else "##" + piece
            if candidate in vocab:
                matched = candidate
                break
            end -= 1
        if matched is None:
            return [unk_token]
        pieces.append(matched)
        start = end
    return pieces

for w in ["unhappiness", "running", "unknownword123"]:
    print(f"{w:16} -> {greedy_wordpiece_segment(w, known_subwords)}")

unhappiness      -> ['[UNK]']
running          -> ['run', '##ning']
unknownword123   -> ['[UNK]']


---
## 3. Word Embeddings

### 3.1 From One-Hot to Distributed Representations

A **one-hot** vector represents each word as a sparse vector of vocabulary size with a
single 1. Problems:
- Dimensionality = vocabulary size (huge).
- **No notion of similarity** — the dot product (cosine similarity) between *any* two
  distinct one-hot vectors is 0, even for near-synonyms like "good"/"great".

**Distributed representations** (dense, low-dimensional vectors, "embeddings") fix this:
similar words get similar vectors, and dimensions jointly encode meaning.

### 3.2 The Distributional Hypothesis

> *"You shall know a word by the company it keeps."* — J. R. Firth

The **distributional hypothesis** states that words appearing in similar contexts tend
to have similar meanings. All embedding methods below are, at their core, ways of
turning word co-occurrence statistics into dense vectors.

In [19]:
vocab_words = ["king", "queen", "man", "woman", "apple", "banana"]
vocab_size = len(vocab_words)

one_hot = np.eye(vocab_size)
print("One-hot vectors:")
for w, v in zip(vocab_words, one_hot):
    print(f"  {w:8} {v}")

from numpy.linalg import norm
def cosine(a, b):
    return a @ b / (norm(a) * norm(b) + 1e-9)

print("\ncos(king, queen) with one-hot  =", cosine(one_hot[0], one_hot[1]))
print("cos(king, apple) with one-hot  =", cosine(one_hot[0], one_hot[4]))
print("-> Identical (both 0): one-hot cannot express that 'king' is closer to 'queen' than 'apple'.")

One-hot vectors:
  king     [1. 0. 0. 0. 0. 0.]
  queen    [0. 1. 0. 0. 0. 0.]
  man      [0. 0. 1. 0. 0. 0.]
  woman    [0. 0. 0. 1. 0. 0.]
  apple    [0. 0. 0. 0. 1. 0.]
  banana   [0. 0. 0. 0. 0. 1.]

cos(king, queen) with one-hot  = 0.0
cos(king, apple) with one-hot  = 0.0
-> Identical (both 0): one-hot cannot express that 'king' is closer to 'queen' than 'apple'.


### 3.3 Word2Vec — CBOW and Skip-gram

**Word2Vec** (Mikolov et al., 2013) learns embeddings by training a shallow neural
network on a simple prediction task, then discarding the task and keeping the learned
weight matrix as embeddings. Two architectures:

- **CBOW (Continuous Bag of Words):** predict the **center word** from its surrounding
  **context words**. Faster to train, works well for frequent words.
- **Skip-gram:** predict the **context words** from the **center word**. Slower, but
  performs better for rare words.

```
CBOW:      [context words] -> center word
Skip-gram:  center word    -> [context words]
```

Both are trained with a sliding window over a large corpus.

#### Negative Sampling
Computing a full softmax over the entire vocabulary for every training step is
expensive. **Negative sampling** reframes training as binary classification: for each
true (center, context) pair, sample a handful of *negative* (center, random-word) pairs
and train the model to distinguish real pairs from noise:

$$\log \sigma(v_{c}^\top v_{w}) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)}\big[\log \sigma(-v_{c}^\top v_{w_i})\big]$$

Negative words are sampled from a **unigram distribution raised to the 3/4 power**,
which up-weights rare words relative to raw frequency.

#### Hierarchical Softmax
An alternative to negative sampling: arrange the vocabulary as leaves of a **binary
(Huffman) tree**. Predicting a word becomes a sequence of binary decisions down the
tree, reducing softmax cost from $O(|V|)$ to $O(\log |V|)$.

#### Subsampling of Frequent Words
Very frequent words ("the", "a") provide little information per occurrence and are
subsampled (probabilistically dropped) during training with probability increasing
with frequency:

$$P(\text{drop } w) = 1 - \sqrt{\frac{t}{f(w)}}$$

where $f(w)$ is the word's corpus frequency and $t$ is a small threshold (e.g. $10^{-5}$).

We train a real Skip-gram model with `gensim` below on a toy corpus.

In [20]:
from gensim.models import Word2Vec

toy_sentences = [
    "the king rules the kingdom".split(),
    "the queen rules the kingdom".split(),
    "the man and woman live in the kingdom".split(),
    "the king and queen love each other".split(),
    "the prince is the son of the king and queen".split(),
    "the princess is the daughter of the king and queen".split(),
    "apple and banana are fruits".split(),
    "the man eats an apple and a banana".split(),
    "the woman eats an apple every day".split(),
] * 30  # repeat for enough training signal on such a tiny corpus

w2v_skipgram = Word2Vec(
    sentences=toy_sentences, vector_size=50, window=3, min_count=1,
    sg=1,            # 1 = skip-gram, 0 = CBOW
    negative=5,       # negative sampling with 5 negative samples
    sample=1e-3,      # subsampling threshold for frequent words
    epochs=100, seed=42,
)

w2v_cbow = Word2Vec(
    sentences=toy_sentences, vector_size=50, window=3, min_count=1,
    sg=0, negative=5, sample=1e-3, epochs=100, seed=42,
)

print("Most similar to 'king' (Skip-gram):", w2v_skipgram.wv.most_similar("king", topn=3))
print("Most similar to 'king' (CBOW)     :", w2v_cbow.wv.most_similar("king", topn=3))
print("\ncos(king, queen) =", w2v_skipgram.wv.similarity("king", "queen"))
print("cos(king, apple) =", w2v_skipgram.wv.similarity("king", "apple"))
print("-> Unlike one-hot vectors, embeddings now reflect semantic closeness.")

Most similar to 'king' (Skip-gram): [('queen', 0.9855800271034241), ('daughter', 0.9166346788406372), ('of', 0.9158729910850525)]
Most similar to 'king' (CBOW)     : [('queen', 0.9893693327903748), ('prince', 0.9502978920936584), ('of', 0.948911190032959)]

cos(king, queen) = 0.98557985
cos(king, apple) = 0.10558749
-> Unlike one-hot vectors, embeddings now reflect semantic closeness.


### 3.4 GloVe — Co-occurrence Matrices & Weighted Least Squares

**GloVe** (Global Vectors, Pennington et al., 2014) takes a different route from
Word2Vec: instead of a predictive neural network, it directly factorizes a **global
word-word co-occurrence matrix** $X$, where $X_{ij}$ counts how often word $j$ appears
in the context of word $i$ across the whole corpus.

GloVe fits word vectors $w_i, \tilde w_j$ and biases $b_i, \tilde b_j$ to satisfy:

$$w_i^\top \tilde w_j + b_i + \tilde b_j \approx \log X_{ij}$$

via a **weighted least squares** objective:

$$J = \sum_{i,j} f(X_{ij}) \left( w_i^\top \tilde w_j + b_i + \tilde b_j - \log X_{ij} \right)^2$$

where $f(X_{ij})$ down-weights very rare *and* very frequent co-occurrences (a capped
power-law weighting function), so extremely common pairs (like "the", "of") don't
dominate the loss.

Below we build a co-occurrence matrix and inspect it (a full GloVe training loop is
just gradient descent on the objective above).

In [21]:
from IPython.display import display
def build_cooccurrence(sentences, window=2):
    cooc = defaultdict(Counter)
    for sent in sentences:
        for i, center in enumerate(sent):
            start, end = max(0, i - window), min(len(sent), i + window + 1)
            for j in range(start, end):
                if j != i:
                    cooc[center][sent[j]] += 1.0 / abs(i - j)  # distance-weighted count
    return cooc

cooc = build_cooccurrence(toy_sentences, window=2)

import pandas as pd
words_of_interest = ["king", "queen", "man", "woman", "kingdom", "apple", "banana"]
matrix = pd.DataFrame(
    [[cooc[w1][w2] for w2 in words_of_interest] for w1 in words_of_interest],
    index=words_of_interest, columns=words_of_interest,
).round(1)
print("Distance-weighted co-occurrence matrix X (this is what GloVe factorizes):")
display(matrix)

def glove_weight(x, x_max=100, alpha=0.75):
    return (x / x_max) ** alpha if x < x_max else 1.0

print("\nGloVe weighting f(X_ij) for a few counts:")
for x in [0.5, 1, 5, 50, 150]:
    print(f"  f({x:6}) = {glove_weight(x):.3f}")

Distance-weighted co-occurrence matrix X (this is what GloVe factorizes):


,king,queen,man,woman,kingdom,apple,banana
king,0.0,45.0,0.0,0.0,0,0.0,0.0
queen,45.0,0.0,0.0,0.0,0,0.0,0.0
man,0.0,0.0,0.0,15.0,0,0.0,0.0
woman,0.0,0.0,15.0,0.0,0,0.0,0.0
kingdom,0.0,0.0,0.0,0.0,0,0.0,0.0
apple,0.0,0.0,0.0,0.0,0,0.0,15.0
banana,0.0,0.0,0.0,0.0,0,15.0,0.0



GloVe weighting f(X_ij) for a few counts:
  f(   0.5) = 0.019
  f(     1) = 0.032
  f(     5) = 0.106
  f(    50) = 0.595
  f(   150) = 1.000


### 3.5 FastText — Character n-grams & Subword Representations

**FastText** (Bojanowski et al., 2017, from Facebook AI) extends Word2Vec's skip-gram
model by representing each word as the **sum of its character n-gram embeddings**,
plus a whole-word vector:

$$\text{word} = \langle\!\langle \text{wh}, \text{whe}, \text{her}, \text{ere}, \text{re}\rangle\!\rangle, \text{re}, \dots \Rightarrow v_{\text{where}} = \sum_g z_g$$

(e.g. `"where"` with n=3 → `<wh, whe, her, ere, re>` plus the special whole-word token
`<where>`, where `<` `>` mark word boundaries).

**Benefits:**
- Can produce embeddings for **OOV / rare / misspelled words** by summing the n-grams
  it *does* recognize — impossible for vanilla Word2Vec/GloVe, which only know whole
  words seen during training.
- Captures morphology well for morphologically rich languages (prefixes/suffixes share
  n-grams across related words, e.g. "run", "running", "runner").

In [22]:
from gensim.models import FastText

ft_model = FastText(
    sentences=toy_sentences, vector_size=50, window=3, min_count=1,
    sg=1, min_n=2, max_n=4,  # character n-gram size range
    epochs=100, seed=42,
)

print("'kingdom' in training vocab?", "kingdom" in ft_model.wv.key_to_index)
print("'kingdoms' in training vocab?", "kingdoms" in ft_model.wv.key_to_index, "(never seen during training)")
print()
print("FastText can still embed the unseen word via character n-grams:")
vec = ft_model.wv["kingdoms"]
print("  vector shape:", vec.shape)
print("  most similar to unseen 'kingdoms':", ft_model.wv.most_similar("kingdoms", topn=3))

try:
    w2v_skipgram.wv["kingdoms"]
except KeyError as e:
    print("\nPlain Word2Vec fails on the same unseen word:", e)

'kingdom' in training vocab? True
'kingdoms' in training vocab? False (never seen during training)

FastText can still embed the unseen word via character n-grams:
  vector shape: (50,)
  most similar to unseen 'kingdoms': [('kingdom', 0.9861388206481934), ('rules', 0.9201077222824097), ('king', 0.893018364906311)]

Plain Word2Vec fails on the same unseen word: "Key 'kingdoms' not present"


---
## 4. Contextual Embeddings

### 4.1 Static vs. Contextual Embeddings

Word2Vec/GloVe/FastText give each word **one fixed vector**, regardless of context:

> "I deposited money in the **bank**." vs. "I sat by the river **bank**."

Both instances of "bank" get the *same* embedding — a clear limitation for polysemous
words. **Contextual embeddings** instead compute a *different* vector for each
occurrence, as a function of the entire sentence.

In [23]:
sentence_a = "I deposited money in the bank".split()
sentence_b = "I sat by the river bank".split()

# Static embeddings: identical vector for 'bank' regardless of context
vec_bank_a = w2v_skipgram.wv["bank"] if "bank" in w2v_skipgram.wv else None
print("With static embeddings, 'bank' would have the SAME vector in both sentences")
print("regardless of the very different meanings ('financial institution' vs 'riverside').")
print("This is exactly the limitation contextual embeddings (ELMo, BERT, GPT) solve.")

With static embeddings, 'bank' would have the SAME vector in both sentences
regardless of the very different meanings ('financial institution' vs 'riverside').
This is exactly the limitation contextual embeddings (ELMo, BERT, GPT) solve.


### 4.2 ELMo — BiLSTM-based Contextualization

**ELMo** (Embeddings from Language Models, Peters et al., 2018) was among the first
widely-used contextual embedding methods. Architecture:

1. A **character-CNN** builds an initial context-*independent* representation for each
   token (handles OOV via characters, like FastText).
2. A **multi-layer bidirectional LSTM** language model processes the sequence — a
   forward LSTM predicting the next word, and a backward LSTM predicting the previous
   word, trained jointly.
3. Each layer's hidden state is a contextualized representation; ELMo's key idea is to
   take a **task-specific weighted sum of all layers** (not just the top layer):

$$\text{ELMo}_k = \gamma \sum_{j=0}^{L} s_j \, h_{k,j}$$

where $h_{k,j}$ is the hidden state for token $k$ at layer $j$, $s_j$ are softmax-normalized
learned weights, and $\gamma$ is a learned scale. Lower layers tend to capture syntax,
higher layers tend to capture semantics — the weighted sum lets a downstream task pick
whichever mixture helps most.

Below is a minimal (untrained, illustrative) BiLSTM run in PyTorch/numpy style logic
to show the *mechanism* of layer aggregation, not to reproduce real ELMo quality.

In [24]:
# Minimal illustration of "layer aggregation" — the core ELMo idea — with random layers
np.random.seed(0)
seq_len, hidden_dim, n_layers = 6, 8, 3

# Pretend these are hidden states from 3 BiLSTM layers for one sentence
layer_states = np.random.randn(n_layers, seq_len, hidden_dim)

# Task-specific softmax-normalized layer weights (normally learned)
raw_weights = np.array([0.2, 0.8, 0.5])
s = np.exp(raw_weights) / np.exp(raw_weights).sum()
gamma = 1.0

elmo_repr = gamma * np.tensordot(s, layer_states, axes=([0], [0]))  # weighted sum over layers

print("Softmax layer weights s_j:", s.round(3))
print("ELMo representation shape (seq_len x hidden_dim):", elmo_repr.shape)
print("\nEach token's final vector mixes syntax-heavy lower layers")
print("and semantics-heavy upper layers, controlled by learned weights s_j.")

Softmax layer weights s_j: [0.24  0.437 0.324]
ELMo representation shape (seq_len x hidden_dim): (6, 8)

Each token's final vector mixes syntax-heavy lower layers
and semantics-heavy upper layers, controlled by learned weights s_j.


### 4.3 BERT — Transformer Encoder Architecture

**BERT** (Bidirectional Encoder Representations from Transformers, Devlin et al., 2018)
replaces ELMo's BiLSTM with a stack of **Transformer encoder layers**. Each layer applies:

1. **Multi-head self-attention** — every token attends to every other token
   (bidirectionally, unlike GPT's causal/left-to-right attention), letting the
   representation of "bank" directly attend to "river" or "money" in the same sentence.
2. **Position-wise feed-forward network** + residual connections + layer normalization.

Because self-attention has no inherent notion of order, BERT adds **positional
embeddings** (and **segment embeddings** to mark sentence A vs. sentence B) to the
input token embeddings before the first layer.

#### Masked Language Modeling (MLM)
BERT is pretrained by randomly masking ~15% of input tokens and training the model to
predict the original token from context on **both sides**:

```
Input : "the [MASK] achieved high accuracy"
Target: predict "model" at the masked position
```
Of the masked positions: 80% become `[MASK]`, 10% are replaced by a random token, and
10% are left unchanged — this prevents the model from only learning to handle the
literal `[MASK]` token, which never appears at fine-tuning/inference time.

#### Next Sentence Prediction (NSP)
BERT's original pretraining also included a **binary classification** task: given two
sentences A and B, predict whether B genuinely follows A in the source text, or is a
random sentence. This encouraged BERT to learn sentence-pair relationships useful for
tasks like natural language inference and QA (later work, e.g. RoBERTa, found NSP's
benefit marginal and dropped it).

#### Transfer Learning & Fine-tuning
BERT is pretrained once on huge unlabeled text (MLM + NSP), producing general-purpose
contextual representations. For a downstream task (classification, NER, QA, ...), a
small task-specific head is added on top and the **whole network is fine-tuned** on a
much smaller labeled dataset — this transfer-learning recipe is why BERT-style models
generalize so well with limited labeled data.

#### Representation Extraction
Two common ways to get a usable representation from BERT:
- **`[CLS]` token** — its final hidden state is often used as a whole-sequence summary
  for classification.
- **Token-level hidden states** — used directly for sequence labeling (NER, POS) or
  pooled (mean/max) for sentence embeddings.

The code below builds a minimal from-scratch Transformer encoder block (attention +
feed-forward) with **NumPy**, so you can see the MLM masking mechanism and self-attention
computation without requiring a network download of pretrained BERT weights.

In [25]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = K.shape[-1]
    scores = (Q @ K.T) / math.sqrt(d_k)          # bidirectional: no causal mask
    attn = softmax(scores, axis=-1)
    return attn @ V, attn

seq_len, d_model = 6, 8
np.random.seed(1)
X = np.random.randn(seq_len, d_model)  # token embeddings + positional embeddings (already summed)
Wq, Wk, Wv = (np.random.randn(d_model, d_model) * 0.1 for _ in range(3))

out, attn_weights = self_attention(X, Wq, Wk, Wv)
tokens = ["[CLS]", "the", "[MASK]", "achieved", "high", "accuracy"]

print("Self-attention weights (row = query token, col = key token it attends to):")
print(pd.DataFrame(attn_weights, index=tokens, columns=tokens).round(2))
print("\nNotice '[MASK]' (row 3) attends to ALL other tokens on both sides —")
print("this bidirectional context is exactly what lets BERT predict the masked word.")

Self-attention weights (row = query token, col = key token it attends to):
          [CLS]   the  [MASK]  achieved  high  accuracy
[CLS]      0.17  0.17    0.16      0.16  0.17      0.17
the        0.16  0.14    0.16      0.15  0.18      0.21
[MASK]     0.17  0.18    0.17      0.17  0.16      0.15
achieved   0.17  0.16    0.17      0.17  0.17      0.17
high       0.17  0.18    0.16      0.16  0.16      0.15
accuracy   0.18  0.19    0.16      0.17  0.16      0.14

Notice '[MASK]' (row 3) attends to ALL other tokens on both sides —
this bidirectional context is exactly what lets BERT predict the masked word.


In [26]:
def mlm_mask_tokens(tokens, mask_prob=0.15, vocab=None, seed=0):
    rng = random.Random(seed)
    vocab = vocab or tokens
    out, labels = [], []
    for tok in tokens:
        if rng.random() < mask_prob:
            labels.append(tok)
            r = rng.random()
            if r < 0.8:
                out.append("[MASK]")
            elif r < 0.9:
                out.append(rng.choice(vocab))   # random token
            else:
                out.append(tok)                  # unchanged
        else:
            out.append(tok)
            labels.append("-")
    return out, labels

sentence_tokens = "the transformer encoder computes contextual token representations".split()
masked, labels = mlm_mask_tokens(sentence_tokens, mask_prob=0.3, seed=7)

print("Original:", sentence_tokens)
print("Masked  :", masked)
print("Labels  :", labels, " (the model is trained to predict the true token wherever a label != '-')")

Original: ['the', 'transformer', 'encoder', 'computes', 'contextual', 'token', 'representations']
Masked  : ['the', '[MASK]', '[MASK]', 'computes', '[MASK]', '[MASK]', '[MASK]']
Labels  : ['-', 'transformer', 'encoder', '-', 'contextual', 'token', 'representations']  (the model is trained to predict the true token wherever a label != '-')


---
## 5. Language Modeling

A **language model (LM)** assigns a probability to a sequence of words:

$$P(w_1, w_2, \dots, w_n)$$

By the **chain rule of probability**, this factorizes into a product of conditional
probabilities:

$$P(w_1, \dots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \dots, w_{i-1})$$

Every LM — n-gram, RNN, or Transformer — is, at its core, a different way of estimating
$P(w_i \mid \text{context})$.

### 5.1 N-gram Language Models & the Markov Assumption

Estimating $P(w_i \mid w_1, \dots, w_{i-1})$ exactly is infeasible (the context grows
unboundedly). The **n-gram Markov assumption** truncates the context to the previous
$n-1$ words:

$$P(w_i \mid w_1, \dots, w_{i-1}) \approx P(w_i \mid w_{i-n+1}, \dots, w_{i-1})$$

For a **bigram** model ($n=2$): $P(w_i \mid w_{i-1}) = \dfrac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}$

This is simple **Maximum Likelihood Estimation (MLE)** from corpus counts.

In [27]:
lm_text_lines = [
    "the model achieved high accuracy on the test set",
    "the model achieved high sensitivity on the validation set",
    "the dataset was collected using wearable sensors",
    "the model was trained on the dataset",
]
lm_corpus = " ".join(lm_text_lines).split()

bigram_counts = Counter(zip(lm_corpus, lm_corpus[1:]))
unigram_counts = Counter(lm_corpus)

def bigram_mle(w1, w2):
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

for w1, w2 in [("the", "model"), ("model", "achieved"), ("the", "dataset")]:
    print(f"P({w2!r} | {w1!r}) = {bigram_mle(w1, w2):.3f}")

P('model' | 'the') = 0.429
P('achieved' | 'model') = 0.667
P('dataset' | 'the') = 0.286


### 5.2 Smoothing Techniques

Raw MLE assigns **zero probability** to any n-gram never seen in training — a single
unseen bigram makes the whole sequence probability zero. **Smoothing** reserves some
probability mass for unseen events.

| Method | Idea |
|---|---|
| **Laplace (add-one)** | Add 1 to every count: $P(w_i\mid w_{i-1}) = \frac{c(w_{i-1},w_i)+1}{c(w_{i-1})+\lvert V\rvert}$. Simple but over-smooths. |
| **Good-Turing** | Re-estimate the probability mass of unseen events from the count of events seen *exactly once* ("count of counts"). |
| **Katz Backoff** | Use the (discounted) higher-order n-gram if seen; otherwise **back off** to a lower-order n-gram, redistributing the discounted mass. |
| **Kneser-Ney** | State of the art among classical smoothing methods. Uses **absolute discounting** plus a "continuation probability" — how many *distinct* contexts a word appears in, rather than raw frequency, better modeling words like "Francisco" that are frequent only after "San". |

In [28]:
V = len(set(lm_corpus))

def laplace_bigram(w1, w2, k=1):
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[w1] + k * V)

def katz_backoff_bigram(w1, w2, discount=0.5):
    if bigram_counts[(w1, w2)] > 0:
        return (bigram_counts[(w1, w2)] - discount) / unigram_counts[w1]
    # back off to unigram probability, roughly redistributing leftover mass
    unigram_prob = unigram_counts[w2] / len(lm_corpus)
    return 0.4 * unigram_prob  # 0.4 = illustrative backoff weight

print(f"{'Bigram':<25}{'MLE':<10}{'Laplace':<10}{'Katz backoff':<12}")
for w1, w2 in [("the", "model"), ("model", "achieved"), ("the", "robot")]:
    mle = bigram_mle(w1, w2) if unigram_counts[w1] else 0.0
    lap = laplace_bigram(w1, w2)
    katz = katz_backoff_bigram(w1, w2)
    print(f"({w1!r}, {w2!r}){'':<8}{mle:<10.3f}{lap:<10.4f}{katz:<12.4f}")

print("\n'the robot' was never seen -> raw MLE gives 0, smoothing still gives a small non-zero probability.")

Bigram                   MLE       Laplace   Katz backoff
('the', 'model')        0.429     0.1667    0.3571      
('model', 'achieved')        0.667     0.1500    0.5000      
('the', 'robot')        0.000     0.0417    0.0000      

'the robot' was never seen -> raw MLE gives 0, smoothing still gives a small non-zero probability.


### 5.3 Neural Language Models

#### Feedforward Neural LM (Bengio et al., 2003)
Concatenate embeddings of the previous $n-1$ words, pass through a feedforward
network, and softmax over the vocabulary to predict the next word. This was the first
neural LM to outperform n-grams, and crucially it *learns* word embeddings jointly —
the direct ancestor of Word2Vec.

#### Recurrent Neural Networks (RNNs)
An RNN maintains a **hidden state** $h_t$ updated at every step, in principle giving it
access to the entire preceding sequence (unlike a fixed n-gram window):

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

In practice, vanilla RNNs struggle with long-range dependencies due to
**vanishing/exploding gradients** during backpropagation through time.

#### LSTM & GRU
**LSTMs** (Hochreiter & Schmidhuber, 1997) add a separate cell state and *gates*
(input, forget, output) that learn what to keep, discard, and output at each step,
mitigating vanishing gradients. **GRUs** (Cho et al., 2014) simplify this with just two
gates (reset, update) — fewer parameters, often comparable performance.

#### Sequence-to-Sequence (Seq2Seq) Models
An **encoder** RNN/LSTM compresses an input sequence into a fixed-size context vector;
a **decoder** RNN/LSTM generates the output sequence conditioned on that vector
(used for translation, summarization, etc.).

#### Attention Mechanisms
Compressing an entire input into one fixed vector is a bottleneck for long sequences.
**Attention** (Bahdanau et al., 2014) lets the decoder, at each output step, compute a
weighted combination over *all* encoder hidden states, focusing on the most relevant
input positions dynamically — this idea is the direct precursor to the Transformer's
self-attention.

#### Transformer Language Models
The **Transformer** (Vaswani et al., 2017) discards recurrence entirely, using only
self-attention + feedforward layers, enabling full parallelization over sequence
positions during training and better long-range modeling. Two dominant families:

- **Autoregressive (decoder-only) models — GPT family**: causal (left-to-right)
  self-attention; trained to predict the next token given all previous tokens:
  $$P(w_i \mid w_1, \dots, w_{i-1})$$
  Natural fit for open-ended text generation.
- **Autoencoding (encoder-only) models — BERT family**: bidirectional self-attention;
  trained with the masked-LM objective (Section 4.3), predicting masked tokens from
  *both* directions of context. Natural fit for representation learning / understanding
  tasks (classification, NER, QA), not free-form generation.

We implement a tiny trigram-based neural LM and a toy autoregressive character LM below
to make the "autoregressive" idea concrete.

In [29]:
# Tiny character-level bigram "autoregressive" LM (illustrative GPT-style generation)
text_for_lm = "the model achieved high accuracy on the validation set " * 10
chars = sorted(set(text_for_lm))
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = {i: c for c, i in char_to_id.items()}

# Build char bigram transition probabilities (this stands in for a trained neural LM)
trans_counts = np.zeros((len(chars), len(chars)))
for c1, c2 in zip(text_for_lm, text_for_lm[1:]):
    trans_counts[char_to_id[c1], char_to_id[c2]] += 1
trans_probs = trans_counts / (trans_counts.sum(axis=1, keepdims=True) + 1e-9)

def generate_autoregressive(seed_char, n_steps=40, temperature=0.7, seed=0):
    rng = np.random.RandomState(seed)
    out = [seed_char]
    current = seed_char
    for _ in range(n_steps):
        probs = trans_probs[char_to_id[current]].copy()
        probs = probs ** (1.0 / temperature)
        probs /= probs.sum()
        next_id = rng.choice(len(chars), p=probs)
        current = id_to_char[next_id]
        out.append(current)
    return "".join(out)

print("Autoregressive generation, one character at a time (each step conditions ONLY on")
print("the previous token(s) - the core idea behind GPT-style decoding):\n")
print(generate_autoregressive("t", n_steps=60))

Autoregressive generation, one character at a time (each step conditions ONLY on
the previous token(s) - the core idea behind GPT-style decoding):

thighel tiethevaccy tion accy mon selighiede the hel vaccurac


---
## 6. Evaluation Metrics

### 6.1 Intrinsic vs. Extrinsic Evaluation

- **Intrinsic evaluation** measures a model/representation on a proxy objective
  disconnected from the final application — e.g. perplexity of a language model, or
  cosine similarity against human word-similarity judgments. Fast and cheap, but a
  good intrinsic score doesn't guarantee good downstream performance.
- **Extrinsic evaluation** measures performance on the actual downstream task
  (translation quality, QA accuracy, classification F1, ...). More expensive but more
  meaningful — it's what ultimately matters.

### 6.2 Perplexity (intrinsic, language modeling)

**Perplexity** measures how "surprised" a language model is by held-out text — lower is
better. For a sequence $w_1, \dots, w_N$:

$$\text{PPL}(W) = P(w_1, \dots, w_N)^{-1/N} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i \mid w_{<i})\right)$$

Perplexity can be interpreted as the **weighted-average branching factor** — roughly,
"how many equally likely choices does the model think it's choosing from, on average,
at each step?"

In [30]:
def perplexity(test_sequence, prob_fn):
    log_probs = []
    for w1, w2 in zip(test_sequence, test_sequence[1:]):
        p = max(prob_fn(w1, w2), 1e-10)  # floor to avoid log(0)
        log_probs.append(math.log(p))
    avg_neg_log_prob = -np.mean(log_probs)
    return math.exp(avg_neg_log_prob)

test_seq = ["the", "model", "achieved", "high", "accuracy"]
ppl_laplace = perplexity(test_seq, laplace_bigram)
print(f"Perplexity of test sequence under the Laplace-smoothed bigram model: {ppl_laplace:.2f}")
print("(Lower perplexity = the model assigns higher probability to the actual held-out text.)")

Perplexity of test sequence under the Laplace-smoothed bigram model: 7.00
(Lower perplexity = the model assigns higher probability to the actual held-out text.)


### 6.3 BLEU, ROUGE, METEOR (extrinsic, generation quality)

These compare a **generated** text against one or more **reference** texts.

- **BLEU** (Bilingual Evaluation Understudy — machine translation): **precision**-oriented.
  Computes n-gram overlap (typically 1- to 4-grams) between candidate and reference,
  geometric-mean-averaged across n, with a **brevity penalty** to discourage overly
  short outputs that could otherwise game precision.
- **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation — summarization):
  **recall**-oriented (ROUGE-N: n-gram recall; ROUGE-L: longest common subsequence).
  Answers "how much of the reference content did the candidate capture?"
- **METEOR**: improves on BLEU by also matching stems, synonyms, and paraphrases (not
  just exact surface n-grams), and combines precision + recall with an explicit
  word-order penalty — historically correlates better with human judgment than BLEU
  at the sentence level.

In [31]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer  # pip install rouge-score

reference = "the cat sat on the mat".split()
candidate_good = "the cat sat on the mat".split()
candidate_bad = "a dog ran in the park".split()

smoothie = SmoothingFunction().method1
print("BLEU (good candidate):", round(sentence_bleu([reference], candidate_good, smoothing_function=smoothie), 3))
print("BLEU (bad candidate) :", round(sentence_bleu([reference], candidate_bad, smoothing_function=smoothie), 3))

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
print("\nROUGE (good candidate):", scorer.score(" ".join(reference), " ".join(candidate_good)))
print("ROUGE (bad candidate) :", scorer.score(" ".join(reference), " ".join(candidate_bad)))

BLEU (good candidate): 1.0
BLEU (bad candidate) : 0.041

ROUGE (good candidate): {'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0)}
ROUGE (bad candidate) : {'rouge1': Score(precision=0.16666666666666666, recall=0.16666666666666666, fmeasure=0.16666666666666666), 'rougeL': Score(precision=0.16666666666666666, recall=0.16666666666666666, fmeasure=0.16666666666666666)}


### 6.4 BERTScore (extrinsic, semantic similarity)

Surface n-gram overlap metrics (BLEU/ROUGE) penalize valid paraphrases that use
different words. **BERTScore** instead embeds each token of the candidate and
reference with a contextual model (e.g. BERT), then greedily matches each candidate
token to its most similar reference token via **cosine similarity**, aggregating into
precision/recall/F1 — this rewards *semantically* equivalent but lexically different
text. (Computing real BERTScore needs a pretrained transformer; we illustrate the
matching mechanism with our toy embeddings from Section 3 instead.)

In [32]:
from numpy.linalg import norm
def bertscore_style(candidate_tokens, reference_tokens, embed_fn):
    def get_vec(tok):
        try:
            return embed_fn(tok)
        except KeyError:
            return np.zeros(50)

    cand_vecs = [get_vec(t) for t in candidate_tokens]
    ref_vecs = [get_vec(t) for t in reference_tokens]

    def cos(a, b):
        return a @ b / (norm(a) * norm(b) + 1e-9)

    # Recall: for each reference token, find best-matching candidate token
    recall = np.mean([max(cos(r, c) for c in cand_vecs) for r in ref_vecs])
    # Precision: for each candidate token, find best-matching reference token
    precision = np.mean([max(cos(c, r) for r in ref_vecs) for c in cand_vecs])
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1

cand = ["king", "man"]
ref = ["queen", "woman"]
p, r, f1 = bertscore_style(cand, ref, lambda t: w2v_skipgram.wv[t])
print(f"BERTScore-style (semantic) match for a paraphrase-like pair:")
print(f"  Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}")
print("  (High despite ZERO exact word overlap -- BLEU/ROUGE would score this ~0.)")

BERTScore-style (semantic) match for a paraphrase-like pair:
  Precision=0.965  Recall=0.965  F1=0.965
  (High despite ZERO exact word overlap -- BLEU/ROUGE would score this ~0.)


### 6.5 Exact Match, Precision, Recall, F1-score

Standard classification-style metrics, widely used for extractive QA, NER, and
classification tasks:

- **Exact Match (EM)**: 1 if the prediction exactly equals the gold answer, else 0
  (common in extractive QA, e.g. SQuAD).
- **Precision** $= \dfrac{TP}{TP + FP}$ — of everything predicted positive, how much
  was correct?
- **Recall** $= \dfrac{TP}{TP + FN}$ — of everything that *should* have been predicted
  positive, how much was found?
- **F1** $= 2 \cdot \dfrac{P \cdot R}{P + R}$ — harmonic mean, balancing precision and
  recall (useful when there's a precision/recall trade-off, e.g. imbalanced classes).

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]   # 1 = entity present, 0 = not
y_pred = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]

print(f"Precision = {precision_score(y_true, y_pred):.3f}")
print(f"Recall    = {recall_score(y_true, y_pred):.3f}")
print(f"F1-score  = {f1_score(y_true, y_pred):.3f}")

# Exact Match for QA-style string answers
gold_answers = ["Paris", "1789", "Isaac Newton"]
pred_answers = ["Paris", "the year 1789", "isaac newton"]

def exact_match(gold, pred):
    return int(gold.strip().lower() == pred.strip().lower())

em_scores = [exact_match(g, p) for g, p in zip(gold_answers, pred_answers)]
print("\nExact Match per example:", em_scores, " -> EM accuracy =", np.mean(em_scores))

Precision = 0.800
Recall    = 0.667
F1-score  = 0.727

Exact Match per example: [1, 0, 1]  -> EM accuracy = 0.6666666666666666


### 6.6 Cosine Similarity (intrinsic, embedding quality)

Already used throughout this notebook (Sections 3–4, 6.4): given two vectors $a, b$,

$$\text{cos}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Ranges from $-1$ (opposite) to $1$ (identical direction), and is the standard way to
compare embeddings, since it's insensitive to vector magnitude — useful because
embedding norms can vary with word frequency without necessarily reflecting semantic
strength.

In [34]:
pairs = [("king", "queen"), ("king", "man"), ("apple", "banana"), ("king", "banana")]
print(f"{'Pair':<20}{'Cosine similarity':<20}")
for a, b in pairs:
    sim = w2v_skipgram.wv.similarity(a, b)
    print(f"{(a + ', ' + b):<20}{sim:<20.3f}")

Pair                Cosine similarity   
king, queen         0.986               
king, man           0.187               
apple, banana       0.913               
king, banana        0.165               


### 6.7 Human Evaluation

Automatic metrics are cheap but imperfect proxies (BLEU/ROUGE penalize valid
paraphrases; perplexity doesn't guarantee factual or coherent generations). **Human
evaluation** asks people to directly rate outputs — common protocols include:

- **Likert-scale ratings** (e.g. 1–5) for fluency, coherence, relevance, factuality.
- **Pairwise/A-B comparison** ("which of these two outputs is better?"), often more
  reliable than absolute scales since it avoids calibration drift between annotators.
- **Best-Worst Scaling**, ranking several candidates at once.
- Task-specific protocols: **adequacy/fluency** for translation, **faithfulness /
  factual consistency** for summarization, **helpfulness / harmlessness** for
  dialogue/assistant outputs.

Human evaluation is the gold standard but is **slow, expensive, and can have
low inter-annotator agreement**, which is why it's typically used to validate — not
replace — automatic metrics like the ones above.

---
## Summary

| Stage | Key Techniques |
|---|---|
| **1. Text Processing** | cleaning, normalization, case folding, stop words, stemming/lemmatization, sentence segmentation, POS tagging, NER, parsing, vectorization |
| **2. Tokenization** | word/char/subword tokenization, BPE, WordPiece, SentencePiece/Unigram LM, vocab & special tokens, OOV handling |
| **3. Word Embeddings** | distributional hypothesis, Word2Vec (CBOW/Skip-gram, negative sampling, hierarchical softmax, subsampling), GloVe, FastText |
| **4. Contextual Embeddings** | static vs. contextual, ELMo (BiLSTM + layer mixing), BERT (Transformer encoder, MLM, NSP, fine-tuning) |
| **5. Language Modeling** | chain rule, n-grams + smoothing (Laplace/Good-Turing/Katz/Kneser-Ney), neural LMs, RNN/LSTM/GRU, seq2seq + attention, Transformer LMs (GPT vs. BERT) |
| **6. Evaluation** | intrinsic vs. extrinsic, perplexity, BLEU/ROUGE/METEOR, BERTScore, EM/P/R/F1, cosine similarity, human evaluation |

Together, these six areas form the classical-to-modern NLP pipeline: clean and
tokenize text, represent it numerically (static → contextual embeddings), model
probability distributions over language, and rigorously evaluate the results.